# Kinyarwanda News Classifier - run everything on Colab
**Before starting:** Runtime -> Change runtime type -> **T4 GPU**.
Run the cells top to bottom.

> **Reproducing this notebook:** it expects the project zip in step 0. Alternatively replace step 0 with:
> `!git clone https://github.com/YOUR_GITHUB_USERNAME/kinyarwanda-news-classifier.git` then `%cd kinyarwanda-news-classifier`.


## 0. Upload the project zip

In [1]:
from google.colab import files
uploaded = files.upload()   # choose kinyarwanda-news-classifier.zip
!unzip -q -o kinyarwanda-news-classifier.zip
%cd kinyarwanda-news-classifier
!ls

Saving kinyarwanda-news-classifier.zip to kinyarwanda-news-classifier.zip
/content/kinyarwanda-news-classifier
app   models	 README.md	     requirements.txt  src
data  notebooks  REPORT_TEMPLATE.md  results


## 1. Install dependencies

In [2]:
!pip install -q -r requirements.txt gradio
!nvidia-smi | head -15

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 9.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
Mon Oct  5 19:23:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  

## 2. Prepare data (about 2-5 min)

In [3]:
!python src/prepare_data.py

Loading KINNEWS (cleaned config)...
README.md: 100% 7.95k/7.95k [00:00<00:00, 11.6MB/s]
kinnews_kirnews.py: 100% 5.37k/5.37k [00:00<00:00, 1.19MB/s]
Generating train split: 100% 17014/17014 [00:01<00:00, 10647.23 examples/s]
Generating test split: 100% 4254/4254 [00:00<00:00, 10792.26 examples/s]
DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 17014
    })
    test: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 4254
    })
})
Labels: ['politics', 'sport', 'economy', 'health', 'entertainment', 'history', 'technology', 'tourism', 'culture', 'fashion', 'religion', 'environment', 'education', 'relationship']
Raw sizes -> train: 17014, test: 4254
Removed 7939 empty/duplicate training rows
Removed 2246 empty/duplicate/leaked test rows

Class distribution:
                train  val  test
label_name                     
culture          156   17    42
economy          954  106   228
education        137   15 

## 2b. Data-quality check (explains how many rows were removed)

In [4]:
!python src/data_quality.py

=== train: 17014 rows ===
empty title:   0
empty content: 0
text shorter than 20 chars: 0
duplicate full texts (extra copies): 7939
duplicate titles (extra copies):     8017
duplicate contents (extra copies):   7956
texts that appear with CONFLICTING labels: 109

=== test: 4254 rows ===
empty title:   0
empty content: 0
text shorter than 20 chars: 0
duplicate full texts (extra copies): 1568
duplicate titles (extra copies):     1575
duplicate contents (extra copies):   1568
texts that appear with CONFLICTING labels: 13

test rows whose exact text also appears in train: 2226
saved results/data_quality.txt


## 3. Baselines (about 2 min)

In [5]:
!python src/baseline.py


=== tfidf_logreg ===
Validation: {'accuracy': 0.804, 'precision_macro': 0.8132, 'recall_macro': 0.6931, 'f1_macro': 0.735, 'f1_weighted': 0.8019}
               precision    recall  f1-score   support

     politics       0.70      0.84      0.76       388
        sport       0.93      0.96      0.94       313
      economy       0.76      0.73      0.75       228
       health       0.74      0.69      0.71       256
entertainment       0.85      0.90      0.87       355
      history       0.77      0.31      0.45        54
   technology       0.92      0.56      0.70        41
      tourism       0.70      0.88      0.78         8
      culture       0.83      0.48      0.61        42
      fashion       0.93      0.65      0.76        20
     religion       0.89      0.88      0.88       131
  environment       0.67      0.38      0.48        16
    education       0.75      0.44      0.56        34
 relationship       0.83      0.88      0.85       122

     accuracy             

## 4. BiLSTM (about 5-10 min)

In [6]:
!python src/bilstm.py --name bilstm --epochs 8

epoch 1 loss 2.0594 | val acc 0.489 f1_macro 0.177
epoch 2 loss 1.4656 | val acc 0.641 f1_macro 0.3304
epoch 3 loss 1.1421 | val acc 0.6696 f1_macro 0.3497
epoch 4 loss 0.9536 | val acc 0.6971 f1_macro 0.3786
epoch 5 loss 0.7891 | val acc 0.6971 f1_macro 0.4131
epoch 6 loss 0.6417 | val acc 0.7059 f1_macro 0.4533
epoch 7 loss 0.5206 | val acc 0.7059 f1_macro 0.4884
epoch 8 loss 0.4003 | val acc 0.7181 f1_macro 0.5559
               precision    recall  f1-score   support

     politics       0.67      0.68      0.67       388
        sport       0.88      0.95      0.91       313
      economy       0.65      0.64      0.64       228
       health       0.54      0.67      0.60       256
entertainment       0.78      0.83      0.80       355
      history       0.46      0.11      0.18        54
   technology       0.60      0.22      0.32        41
      tourism       0.41      0.88      0.56         8
      culture       0.55      0.29      0.38        42
      fashion       0.83    

## 5. Fine-tune AfriBERTa (about 20-40 min)
If it runs out of memory, use `--batch 8`. If it is too slow, add `--max_train 8000`.

In [7]:
!python src/finetune.py --model castorini/afriberta_base --name afriberta_base --lr 3e-5 --epochs 3 --max_len 256

config.json: 100% 729/729 [00:00<00:00, 2.61MB/s]
tokenizer_config.json: 100% 257/257 [00:00<00:00, 1.15MB/s]

sentencepiece.bpe.model: downloading bytes:   0% 0.00/1.55M [00:00<?, ?B/s]
sentencepiece.bpe.model: downloading bytes: 100% 1.12M/1.12M [00:01<00:00, 764kB/s,  108kB/s  ]
sentencepiece.bpe.model: reconstructing file: 100% 1.55M/1.55M [00:01<00:00, 1.06MB/s,  150kB/s  ]
special_tokens_map.json: 100% 150/150 [00:00<00:00, 556kB/s]
Map: 100% 8167/8167 [00:09<00:00, 880.52 examples/s]
Map: 100% 908/908 [00:00<00:00, 994.97 examples/s] 
Map: 100% 2008/2008 [00:02<00:00, 845.74 examples/s]

pytorch_model.bin: downloading bytes:  42% 189M/446M [00:02<00:01, 178MB/s, 13.6MB/s  ]
pytorch_model.bin: downloading bytes:  51% 225M/446M [00:02<00:01, 179MB/s, 19.5MB/s  ]
pytorch_model.bin: reconstructing file:  50% 222M/446M [00:02<00:02, 88.9MB/s, 12.5MB/s  ]
pytorch_model.bin: downloading bytes:  64% 288M/446M [00:03<00:01, 85.5MB/s, 21.2MB/s  ]
pytorch_model.bin: downloading bytes:  94%

## 6. Extra experiments (each adds a row to your results table)

In [8]:
!python src/finetune.py --model castorini/afriberta_base --name afriberta_base_len128 --lr 3e-5 --epochs 3 --max_len 128
!python src/finetune.py --model castorini/afriberta_base --name afriberta_base_lr2e5 --lr 2e-5 --epochs 4 --max_len 256

Map: 100% 8167/8167 [00:06<00:00, 1188.31 examples/s]
Map: 100% 908/908 [00:00<00:00, 1149.88 examples/s]
Map: 100% 2008/2008 [00:02<00:00, 797.41 examples/s]
Loading weights: 100% 133/133 [00:00<00:00, 10203.81it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: castorini/afriberta_base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.decoder.bias       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.decoder.weight     | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING

## 7. Results table and error analysis

In [9]:
import pandas as pd
pd.set_option('display.max_colwidth', 100)
pd.read_csv('results/experiments.csv')

,model,config,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted
0,tfidf_logreg,"TF-IDF 1-2gram 50k, LogReg C=10",0.8023,0.8045,0.6832,0.7215,0.7962
1,tfidf_nb,"TF-IDF 1-2gram 50k, MultinomialNB alpha=0.1",0.7585,0.8317,0.5293,0.5787,0.7416
2,bilstm,"emb=200, hid=128, maxlen=256, epochs=8, lr=0.001, w2v=False",0.7082,0.7048,0.5225,0.5239,0.6916
3,afriberta_base,"castorini/afriberta_base, lr=3e-05, epochs=3, max_len=256, batch=16, n_train=8167, class_weights...",0.7779,0.6845,0.6969,0.6761,0.7736
4,afriberta_base_len128,"castorini/afriberta_base, lr=3e-05, epochs=3, max_len=128, batch=16, n_train=8167, class_weights...",0.7585,0.6541,0.6541,0.6407,0.7536
5,afriberta_base_lr2e5,"castorini/afriberta_base, lr=2e-05, epochs=4, max_len=256, batch=16, n_train=8167, class_weights...",0.7784,0.6867,0.7043,0.6811,0.7733


In [10]:
!python src/error_analysis.py --name afriberta_base

               precision    recall  f1-score   support

      culture       0.66      0.55      0.60        42
      economy       0.70      0.70      0.70       228
    education       0.57      0.62      0.59        34
entertainment       0.88      0.87      0.87       355
  environment       0.38      0.38      0.38        16
      fashion       0.72      0.65      0.68        20
       health       0.68      0.68      0.68       256
      history       0.62      0.24      0.35        54
     politics       0.74      0.73      0.74       388
 relationship       0.81      0.90      0.86       122
     religion       0.78      0.86      0.82       131
        sport       0.93      0.97      0.95       313
   technology       0.68      0.61      0.64        41
      tourism       0.44      1.00      0.62         8

     accuracy                           0.78      2008
    macro avg       0.68      0.70      0.68      2008
 weighted avg       0.78      0.78      0.77      2008


Errors

## 8. Upload the model to Hugging Face
Create a token at huggingface.co/settings/tokens (role: **write**). Replace YOUR_USERNAME.

In [11]:
from huggingface_hub import notebook_login
notebook_login()

In [12]:
!python src/upload_to_hub.py --folder models/afriberta_base --repo Seziberagabriel/kinyarwanda-news-classifier

Found 5 files to upload
  Preparing   ████████████████████  5 / 5 ✓
  Uploading   ████████████████████  -
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 5
  Preparing   ████████████████████  5 / 5 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 5
  Preparing   ████████████████████  5 / 5 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 5
  Preparing   ████████████████████  5 / 5 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 5
  Preparing   ████████████████████  5 / 5 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 5
  Preparing   ████████████████████  5 / 5 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files  559kB · 1.12MB/s
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 5
  Preparing   ████████████████████  5 / 5 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files  2.24MB · 1.79MB/s
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 5
  Preparing

## 9. Download your results

In [13]:
!zip -qr results.zip results
from google.colab import files
files.download('results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>